# 1: Loading and Inspecting the Data

In [10]:
import pandas as pd

In [ ]:
# load dataset
df = pd.read_csv("HealthConnect_Appointment_Data.csv")

# view  first 5 rows
print(df.head())

# checking row and columns
print(df.shape)


  appointment_id patient_id  gender  age age_group         appointment_type  \
0       HC-00001     P-1613  Female   39     35-44                Follow-up   
1       HC-00002     P-0813    Male   31     25-34  Specialist Consultation   
2       HC-00003     P-1366  Female   50     45-54     General Consultation   
3       HC-00004     P-1031    Male   59     55-64                Follow-up   
4       HC-00005     P-1458  Female   34     25-34                Follow-up   

  booking_date appointment_date appointment_day appointment_time  \
0     2/6/2025        2/18/2025         Tuesday        Afternoon   
1    2/25/2026        2/27/2026          Friday          Morning   
2   11/16/2025       12/24/2025       Wednesday          Morning   
3    7/18/2025        8/28/2025        Thursday          Evening   
4     7/9/2025        8/25/2025          Monday        Afternoon   

   booking_lead_days  previous_appointments  previous_no_shows reminder_sent  \
0                 12                

In [ ]:
# checkinging data types and summary
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   appointment_id         5000 non-null   object 
 1   patient_id             5000 non-null   object 
 2   gender                 5000 non-null   object 
 3   age                    5000 non-null   int64  
 4   age_group              5000 non-null   object 
 5   appointment_type       5000 non-null   object 
 6   booking_date           5000 non-null   object 
 7   appointment_date       5000 non-null   object 
 8   appointment_day        5000 non-null   object 
 9   appointment_time       5000 non-null   object 
 10  booking_lead_days      5000 non-null   int64  
 11  previous_appointments  5000 non-null   int64  
 12  previous_no_shows      5000 non-null   int64  
 13  reminder_sent          5000 non-null   object 
 14  reminder_channel       3634 non-null   object 
 15  dist

# 2 : Checking for Missing Values

In [ ]:
# checking missing values per column
print(df.isnull().sum())

appointment_id              0
patient_id                  0
gender                      0
age                         0
age_group                   0
appointment_type            0
booking_date                0
appointment_date            0
appointment_day             0
appointment_time            0
booking_lead_days           0
previous_appointments       0
previous_no_shows           0
reminder_sent               0
reminder_channel         1366
distance_to_clinic_km      90
waiting_time_minutes       60
appointment_outcome         0
dtype: int64


# 3: Logic and Quality checks

1. Date Consistency checks

In [29]:
# convert to datetime
df["booking_date"] = pd.to_datetime(df["booking_date"])
df["appointment_date"] = pd.to_datetime(df["appointment_date"])

# 1. checking if appointment date is before booking date
invalid_dates = df[df["appointment_date"] < df["booking_date"]]
print(f"Invalid date order records: {len(invalid_dates)}")

# 2. checking if lead days match the calculated difference
calculated_lead_days = (df["appointment_date"] - df["booking_date"]).dt.days
mismatched_lead_days = df[df["booking_lead_days"] != calculated_lead_days]
print(f"Mismatched lead day records: {len(mismatched_lead_days)}")



Invalid date order records: 0
Mismatched lead day records: 0


2. Patient History Logic checks

In [30]:
# checking if previous_no_shows > previous_appointments
invalid_history = df[df["previous_no_shows"] > df["previous_appointments"]]
print(f"Invalid patient history records: {len(invalid_history)}")

Invalid patient history records: 0


3. Reminder Logic checks

In [31]:
#  records where reminder_sent is 'No' but channel is not 'None'
invalid_no_reminder = df[
    (df["reminder_sent"] == "No") & (df["reminder_channel"] != "None")
]
print(
    f"Invalid 'No reminder' records: {len(invalid_no_reminder)}"
)

# records where reminder_sent is 'Yes' but has an invalid channel
valid_channels = ["SMS", "Email", "WhatsApp"]
invalid_yes_reminder = df[
    (df["reminder_sent"] == "Yes") & (~df["reminder_channel"].isin(valid_channels))
]
print(
    f"Invalid 'Yes reminder' records: {len(invalid_yes_reminder)}"
)


Invalid 'No reminder' records: 1366
Invalid 'Yes reminder' records: 0


4. Age and Age Group Validation

In [33]:
# 1. checking for patients under 18
underage_patients = df[df["age"] < 18]
print(f"Underage records: {len(underage_patients)}")

# 2. Function to derive expected age group
def get_age_group(age):
    if 18 <= age <= 24:
        return "18-24"
    elif 25 <= age <= 34:
        return "25-34"
    elif 35 <= age <= 44:
        return "35-44"
    elif 45 <= age <= 54:
        return "45-54"
    elif 55 <= age <= 64:
        return "55-64"
    elif age >= 65:
        return "65+"
    return "Invalid"

# Compare actual age group with expected age group
df["expected_age_group"] = df["age"].apply(get_age_group)
mismatched_age_groups = df[df["age_group"] != df["expected_age_group"]]
print(f"Mismatched age group records: {len(mismatched_age_groups)}")


Underage records: 0
Mismatched age group records: 0


5. Outcome Categories Check

In [34]:
allowed_outcomes = {"Attended", "No-Show", "Cancelled"}
actual_outcomes = set(df["appointment_outcome"].dropna().unique())

unexpected_outcomes = actual_outcomes - allowed_outcomes
print(f"Unique outcomes found: {actual_outcomes}")
print(f"Unexpected outcome values: {unexpected_outcomes}")

Unique outcomes found: {'Cancelled', 'No-Show', 'Attended'}
Unexpected outcome values: set()


# 4: Summary Statistics


In [37]:
# summary for numbers (age, lead days, distance, waiting time)
print(df.describe())

               age                booking_date            appointment_date  \
count  5000.000000                        5000                        5000   
mean     48.794800  2025-08-31 03:01:09.120000  2025-09-29 18:20:44.160000   
min      18.000000         2024-11-07 00:00:00         2025-01-01 00:00:00   
25%      33.000000         2025-04-15 00:00:00         2025-05-15 00:00:00   
50%      49.000000         2025-08-28 00:00:00         2025-09-27 00:00:00   
75%      64.000000         2026-01-16 06:00:00         2026-02-15 00:00:00   
max      80.000000         2026-06-27 00:00:00         2026-06-30 00:00:00   
std      18.138547                         NaN                         NaN   

       booking_lead_days  previous_appointments  previous_no_shows  \
count         5000.00000            5000.000000        5000.000000   
mean            29.63860               3.013800           0.544200   
min              0.00000               0.000000           0.000000   
25%             1

In [38]:
# breakdown of reminder channels
print(df["reminder_channel"].value_counts())

reminder_channel
SMS         2000
WhatsApp    1101
Email        533
Name: count, dtype: int64


In [36]:
# breakdown of appointment outcomes
print(df["appointment_outcome"].value_counts(normalize=True) * 100)

appointment_outcome
No-Show      48.46
Attended     46.28
Cancelled     5.26
Name: proportion, dtype: float64
